# EchoLens — Nepal GLOF (EMSR927) on Colab

Downloads the NISAR GCOV pair `048_D_074` (**16 Aug → 28 Aug 2026**, ~7 GB × 2) when the laptop is full.

Needs a free Earthdata account. Do **not** paste passwords into the notebook; Colab will prompt.

Runtime: pick a standard CPU machine (disk is what matters, ~80 GB).

In [ ]:
%pip install -q earthaccess h5py numpy matplotlib pyproj scipy shapely rasterio

In [ ]:
import earthaccess

auth = earthaccess.login(strategy="interactive", persist=True)
print("authenticated", bool(getattr(auth, "authenticated", False)))

In [ ]:
from pathlib import Path

NEEDLE = "048_D_074"
BBOX = (85.10, 27.90, 85.55, 28.35)
WINDOWS = (
    ("before", "2026-08-14", "2026-08-18"),
    ("after", "2026-08-27", "2026-08-30"),
)
SHORT_NAME = "NISAR_L2_GCOV_PROVISIONAL_V1"
OUT = Path("/content/nepal/gcov")
OUT.mkdir(parents=True, exist_ok=True)


def granule_id(item) -> str:
    umm = item.get("umm", {}) if hasattr(item, "get") else {}
    return str(umm.get("GranuleUR") or item)


chosen = []
for label, start, end in WINDOWS:
    hits = earthaccess.search_data(
        short_name=SHORT_NAME,
        bounding_box=BBOX,
        temporal=(start, end),
        count=20,
    )
    match = [g for g in hits if NEEDLE in granule_id(g) and "GCOV" in granule_id(g)]
    print(f"{label}: {len(match)} match {NEEDLE}")
    for g in match:
        print(" ", granule_id(g), "~", getattr(g, "size", "?"), "MB")
    if not match:
        raise SystemExit(f"no {NEEDLE} in {label}")
    chosen.append(match[0])

paths = earthaccess.download(chosen, local_path=str(OUT), threads=2, show_progress=True)
for p in paths:
    fp = Path(p)
    print("saved", fp, f"{fp.stat().st_size/1e9:.2f} GB" if fp.exists() else "MISSING")

Optional: copy the `.h5` files to Drive (needs ~14 GB free there). Skip this if you only need the small crop.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/echolens/nepal/gcov"
# !cp -v /content/nepal/gcov/*.h5 "/content/drive/MyDrive/echolens/nepal/gcov/"

The laptop map already uses a Rasuwa crop (`scripts/crop_nepal.py`). To rebuild that crop here without keeping 14 GB:

In [ ]:
import h5py
import numpy as np
from pyproj import Transformer

GLOF_BBOX = (85.08, 27.90, 85.42, 28.30)
GRID = "science/LSAR/GCOV/grids/frequencyB"
POLS = ("HHHH", "HVHV")
CROP = Path("/content/nepal/crop")
CROP.mkdir(parents=True, exist_ok=True)

files = sorted(OUT.glob("*.h5"))
print("h5", [p.name for p in files])

dated = []
for path in files:
    with h5py.File(path, "r") as f:
        start = f["science/LSAR/identification/zeroDopplerStartTime"][()]
        if isinstance(start, bytes):
            start = start.decode()
        dated.append((start[:10], path))
dated.sort()


def extract(path: Path, out: Path, date: str) -> None:
    with h5py.File(path, "r") as f:
        g = f[GRID]
        epsg = int(g["projection"][()])
        x = g["xCoordinates"][:]
        y = g["yCoordinates"][:]
        dx = float(g["xCoordinateSpacing"][()])
        dy = float(g["yCoordinateSpacing"][()])
        to_utm = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
        west, south = to_utm.transform(GLOF_BBOX[0], GLOF_BBOX[1])
        east, north = to_utm.transform(GLOF_BBOX[2], GLOF_BBOX[3])
        col = np.where((x >= west) & (x <= east))[0]
        row = np.where((y >= south) & (y <= north))[0]
        rows = slice(int(row.min()), int(row.max()) + 1)
        cols = slice(int(col.min()), int(col.max()) + 1)
        payload = {
            "epsg": np.int32(epsg),
            "x": x[cols],
            "y": y[rows],
            "dx": np.float64(dx),
            "dy": np.float64(dy),
            "date": np.array(date),
            "granule": np.array(path.name),
            "pols": np.array([p for p in POLS if p in g]),
            "mask": g["mask"][rows, cols],
        }
        for pol in payload["pols"]:
            payload[str(pol)] = g[str(pol)][rows, cols]
    np.savez_compressed(out, **payload)
    print("wrote", out, f"{out.stat().st_size/1e6:.1f} MB")


extract(dated[0][1], CROP / "before.npz", dated[0][0])
extract(dated[-1][1], CROP / "after.npz", dated[-1][0])

In [ ]:
from google.colab import files

files.download("/content/nepal/crop/before.npz")
files.download("/content/nepal/crop/after.npz")